In [1]:
from gpt_functions import get_current_time, tools

tools에 넣는 딕셔너리는 실제 함수를 담는 게 아니라, LLM에게 "이런 함수를 사용할 수 있어"라고 알려주는 설명서(schema)

왜 딕셔너리인가?

OpenAI API 같은 곳에서는 tools를 JSON 형태로 전달하기 때문.

``` txt
tools
 └─ 리스트
     └─ 딕셔너리
         ├─ type: "function"
         └─ function
             ├─ name: "get_current_time"
             └─ description: "현재 날짜와 시간을 반환합니다."
```

> "AI야, 네가 사용할 수 있는 도구가 하나 있는데 이름은 get_current_time이고, 현재 시간을 알려주는 함수야."

``` txt
사용자
 ↓
"현재 시간 알려줘"
 ↓
LLM
 ↓
"get_current_time 함수를 사용해야겠다"
 ↓
tool call 생성
 ↓
Python 프로그램이 get_current_time() 실행
 ↓
"2026-08-30 16:15:32"
 ↓
그 결과를 다시 LLM에게 전달
 ↓
LLM이 사용자에게 답변
```

> tools의 딕셔너리 = LLM에게 제공하는 "함수 사용 설명서"
> get_current_time() = 실제 Python 함수 실행.

In [2]:
if __name__ == '__main__':
    get_current_time('America/New_York')

2026-08-30 05:10:26 America/New_York


## what time is it?

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [4]:
def get_ai_response(messages, tools=None):
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=messages,
        tools=tools,
        reasoning_effort="none",
    )
    return response  # 생성된 응답 내용 반환

In [5]:
messages = [
    {"role": "system", "content": "너는 사용자를 도와주는 상담사야."},  # 초기 시스템 메시지
]

In [6]:
messages = []

while True:
    user_input = input("사용자\t: ")  # 사용자 입력 받기

    if user_input == "exit":  # 사용자가 대화를 종료하려는지 확인
        break

    print("사용자\t: " + user_input)
    messages.append({"role": "user", "content": user_input})  # 사용자 메시지 대화 기록에 추가

    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    messages.append(ai_message)  # tool_calls를 가진 assistant 메시지를 먼저 추가

    print(ai_message)  # gpt에서 반환되는 값을 파악하기 위해 임시로 추가

    tool_calls = ai_message.tool_calls  # AI 응답에 포함된 tool_calls를 가져옵니다.
    if tool_calls:  # tool_calls가 있는 경우
        for tool_call in tool_calls:
            tool_name = tool_call.function.name  # 실행해야한다고 판단한 함수명 받기
            tool_call_id = tool_call.id  # tool_call 아이디 받기
            arguments = json.loads(tool_call.function.arguments)  # 문자열을 딕셔너리로 변환

            if tool_name == "get_current_time":  # 만약 tool_name이 "get_current_time"이라면
                messages.append({
                    "role": "tool",  # role을 "tool"으로 설정
                    "tool_call_id": tool_call_id,
                    # "name": tool_name,
                    "content": get_current_time(timezone=arguments['timezone']),  # 타임존 추가
                })

        messages.append({"role": "system", "content": "이제 주어진 결과를 바탕으로 답변할 차례다."})  # 함수 실행 완료 메시지 추가
        ai_response = get_ai_response(messages, tools=tools)  # 다시 GPT 응답 받기
        ai_message = ai_response.choices[0].message
        messages.append(ai_message)  # 최종 AI 응답을 대화 기록에 추가하기

    print("AI\t: " + (ai_message.content or ""))  # AI 응답 출력

사용자	: 뉴욕, 런던, 파리 시간 알려줘
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_YjarKzbyP8KFKBXycYjkJCZi', function=Function(arguments='{"timezone": "America/New_York"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_0XMfAOXNTI6AIs676xn1gtP4', function=Function(arguments='{"timezone": "Europe/London"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_2ogUk5WaPC8K1NXDsPoCY223', function=Function(arguments='{"timezone": "Europe/Paris"}', name='get_current_time'), type='function')])
2026-08-30 05:10:32 America/New_York
2026-08-30 10:10:32 Europe/London
2026-08-30 11:10:32 Europe/Paris
AI	: 현재 시간은 다음과 같습니다.

- **뉴욕:** 2026년 8월 30일 오전 5:10
- **런던:** 2026년 8월 30일 오전 10:10
- **파리:** 2026년 8월 30일 오전 11:10


> tool_call.id는 함수 자체의 ID가 아니라, 매번 발생하는 “함수 호출 요청” 하나를 식별하는 고유 ID

### 타임존 정보를 받기 전 get_current_time 함수 호출 결과
```txt
사용자	: 안녕?
ChatCompletionMessage(content='안녕하세요! 무엇을 도와드릴까요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
AI	: 안녕하세요! 무엇을 도와드릴까요?
사용자	: 지금 시간은 몇시야?
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XnWCfBXzi6GCHHY7dSgnEKi3', function=Function(arguments='{}', name='get_current_time'), type='function')])
2026-08-30 17:02:39
AI	: 현재 **오후 5시 2분**입니다.
사용자	: 지금 뉴욕 시간은 몇시야?
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_MamsGH34v8esvG8YqRVqvseG', function=Function(arguments='{}', name='get_current_time'), type='function')])
2026-08-30 17:02:57
AI	: 뉴욕은 현재 **오전 4시 2분**입니다. (EDT, 한국보다 13시간 느림)
```

### 함수를 차례로 실행 할 수 없도록 하면 응답을 제대로 못하는 이유
- 뉴욕, 런던, 파리는 서로 다른 3개의 시간 조회 작업이라서 모델이 get_current_time을 3번 호출해야 함.
- 여러 번 실행을 허용하지 않으면 한 번의 tool call만 처리하고 나머지 도시의 시간은 조회하지 못해서 완전한 답변을 만들 수 없음.
- “파라미터가 여러 개”라서가 아닌 “하나의 사용자 요청 안에 여러 개의 독립적인 함수 실행이 필요하기 때문”